This is a notebook for computing deep foundation axial geotechnical capacity. <br>
Author: Zhiyan Jiang [(linkedin.com/in/zhiyanjiang)](http://www.linkedin.com/in/zhiyanjiang)

In [27]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
#import numpy as np
import yaml

In [29]:
from deep_foundation_bearing_capacity.cross_sections.cross_sections import CircularSection, CrossSection
from deep_foundation_bearing_capacity.factor_of_safety.factor_of_safety import FactorOfSafetyDeepFoundation
from deep_foundation_bearing_capacity.foundation.deep_foundation import DeepFoundation
from deep_foundation_bearing_capacity.segments.segments import Segment
from deep_foundation_bearing_capacity.segments.unit_resistance import EndResistance, SideResistance
from deep_foundation_bearing_capacity.soil_layer.layer import Layer
from deep_foundation_bearing_capacity.soil_layer.soil import Soil

In [30]:
# Read soil parameters from YAML
with open("data/soil_params.yaml") as f:
    soils = yaml.safe_load(f)

soil_obj_list = []
for soil in soils:
    soil_obj = Soil.from_dict(soil)
    soil_obj_list.append(soil_obj)

In [ ]:
# Read layer parameters from YAML

with open("data/layer_params.yaml") as f:
    layers = yaml.safe_load(f)


layer_obj_list = []
for layer in layers:

    layer_obj = Layer(soil_obj_list[layer["soil_index"]], layer["ground_water_depth"], layer["top_depth"], layer["thickness"])
    layer_obj_list.append(layer_obj)

    print(f"Effective vertical stress is: {layer_obj.effective_stress_mid}")


ValueError: too many values to unpack (expected 2)

In [ ]:
# test effective_stress_mid
print(f"First layer effective vertical stress at mid is: {layer_obj_list[0].effective_stress_mid}")

In [ ]:
# test SideResistance class for cohesionless layer
side_resistance_obj_1 = SideResistance(layer_obj_list[0])
print(f"Top layer beta is: {side_resistance_obj_1._calculate_beta():.2f}")
print(f"Top layer unit side resistance is: {side_resistance_obj_1.side_resistance_unit():.2f}")

In [ ]:
# test SideResistance class for cohesive layer
side_resistance_obj_2 = SideResistance(layer_obj_list[1])
print(f"Second layer alpha is: {side_resistance_obj_2._calculate_alpha():.2f}")
print(f"Second layer unit side resistance is: {side_resistance_obj_2.side_resistance_unit():.2f}")

In [ ]:
# test EndResistance class for cohesionless layer
end_resistance_obj_1 = EndResistance(layer_obj_list[0])
print(f"Top layer unit end resistance is: {end_resistance_obj_1.end_resistance_unit():.2f}")

In [ ]:
# test EndResistance class for cohesionless layer
end_resistance_obj_2 = EndResistance(layer_obj_list[1])
print(f"Second layer unit end resistance is: {end_resistance_obj_2.end_resistance_unit():.2f}")

In [ ]:
# test CrossSection class
cross_section_diameter = 1.5
print(f"Cross section diameter is: {cross_section_diameter}")
cross_section_obj = CircularSection(1.5)
print(f"Cross section area is: {cross_section_obj._cross_section_area:.2f}")
print(f"Cross section perimeter is: {cross_section_obj._perimeter:.2f}")

In [ ]:
# test FactorOfSafety class
fs = 3.0
print(f"Factor of safety is prescribed as: {fs}")
fs_obj = FactorOfSafetyDeepFoundation(3, 3)

In [ ]:
# test Segment class
segment_obj_1 = Segment(cross_section_obj, layer_obj_list[0], fs_obj)
print(f"Top segment side surface area is: {segment_obj_1._side_surface_area:.2f}")


In [ ]:
# test sgement SideResistance for cohesionless layer
print(f"Top segment side resistance is: {segment_obj_1.calculate_side_resistance()}")
print(f"Top segment side resistance is: {segment_obj_1.side_resistance}")

# test segment EndResistance class for cohesionless layer
print(f"Top segment end resistance is: {segment_obj_1.calculate_end_resistance()}")
print(f"Top segment end resistance is: {segment_obj_1.end_resistance}")

In [ ]:
# test segment EndResistance class for cohesive layer
segment_obj_2 = Segment(cross_section_obj, layer_obj_list[1], fs_obj)
print(f"Second segment side resistance is: {segment_obj_2.calculate_side_resistance()}")
print(f"Second segment end resistance is: {segment_obj_2.calculate_end_resistance()}")


In [ ]:
# test DeepFoundation class
top_depth = 0
deep_foundation_obj = DeepFoundation([segment_obj_1, segment_obj_2], top_depth, resistance_corrections = None)
print(f"Without depth correction, second segment side resistance is: {deep_foundation_obj.segments[1].side_resistance}")



In [ ]:
# test DeepFoundation accumulative side resistance 
deep_foundation_obj.calculate_segments_side_resistances_accumulative()

In [ ]:
# Test side resistance correction
#deep_foundation_obj = DeepFoundation([segment_obj_1, segment_obj_2], top_depth, resistance_correction=True)
#print(f"With depth correction, second segment side resistance is: {deep_foundation_obj.segments[1].side_resistance}")